# Traffic Light Violation Detection
YOLOv11 + SORT Kalman Filter + HSV Traffic Light + Violation Logic

## 1. Install Dependencies

In [43]:
!pip install ultralytics filterpy scipy -q
!pip install tqdm -q

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
import os
os.makedirs('/content/drive/MyDrive/finpro-pkac/output', exist_ok=True)

## 2. Imports

In [46]:
import cv2
import numpy as np
from ultralytics import YOLO
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment
from collections import deque
from tqdm import tqdm
import torch

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


## 3. Vehicle Detector (YOLOv11)

In [47]:
class VehicleDetector:
    VEHICLE_CLASSES = {1, 2, 3, 5, 7}

    def __init__(self, model_path='yolo11m.pt', conf=0.35, iou=0.4, device='cuda'):
        self.model = YOLO(model_path)
        self.conf = conf
        self.iou = iou
        self.device = device

    def detect(self, frame, polygon=None):
      results = self.model(
          frame,
          conf=self.conf,
          iou=self.iou,
          device=self.device,
          verbose=False,
          classes=list(self.VEHICLE_CLASSES)
      )[0]

      boxes = []
      for box in results.boxes:
          cls_id = int(box.cls[0])
          if cls_id not in self.VEHICLE_CLASSES:
              continue
          x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
          score = float(box.conf[0].cpu())

          if polygon is not None:
              cx = (x1 + x2) / 2
              cy = (y1 + y2) / 2
              result = cv2.pointPolygonTest(
                  polygon.reshape((-1, 1, 2)).astype(np.float32),
                  (float(cx), float(cy)),
                  False
              )
              if result < 0:
                  continue

          boxes.append([x1, y1, x2, y2, score])

      boxes = np.array(boxes) if boxes else np.empty((0, 5))
      boxes = self._nms(boxes)
      return boxes

    def _nms(self, boxes, iou_thresh=0.5):
        if len(boxes) == 0:
            return boxes
        scores = boxes[:, 4]
        order = scores.argsort()[::-1]
        keep = []
        suppressed = set()
        for i in order:
            if i in suppressed:
                continue
            keep.append(i)
            for j in order:
                if j <= i or j in suppressed:
                    continue
                if self._iou(boxes[i], boxes[j]) > iou_thresh:
                    suppressed.add(j)
        return boxes[keep]

    def _iou(self, a, b):
        ax1, ay1, ax2, ay2 = a[:4]
        bx1, by1, bx2, by2 = b[:4]
        ix1 = max(ax1, bx1)
        iy1 = max(ay1, by1)
        ix2 = min(ax2, bx2)
        iy2 = min(ay2, by2)
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
        area_a = (ax2 - ax1) * (ay2 - ay1)
        area_b = (bx2 - bx1) * (by2 - by1)
        union = area_a + area_b - inter
        return inter / union if union > 0 else 0

## 4. SORT Tracker (Kalman Filter)

In [48]:
class KalmanBoxTracker:
    count = 0

    def __init__(self, bbox):
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([
            [1,0,0,0,1,0,0],
            [0,1,0,0,0,1,0],
            [0,0,1,0,0,0,1],
            [0,0,0,1,0,0,0],
            [0,0,0,0,1,0,0],
            [0,0,0,0,0,1,0],
            [0,0,0,0,0,0,1]
        ], dtype=float)
        self.kf.H = np.array([
            [1,0,0,0,0,0,0],
            [0,1,0,0,0,0,0],
            [0,0,1,0,0,0,0],
            [0,0,0,1,0,0,0]
        ], dtype=float)
        self.kf.R[2:, 2:] *= 10.0
        self.kf.P[4:, 4:] *= 1000.0
        self.kf.P *= 10.0
        self.kf.Q[-1, -1] *= 0.01
        self.kf.Q[4:, 4:] *= 0.01
        self.kf.x[:4] = self._to_z(bbox)
        self.id = KalmanBoxTracker.count
        KalmanBoxTracker.count += 1
        self.hits = 1
        self.hit_streak = 1
        self.age = 0
        self.time_since_update = 0

    def _to_z(self, bbox):
        x1, y1, x2, y2 = bbox
        w = x2 - x1
        h = y2 - y1
        cx = x1 + w / 2
        cy = y1 + h / 2
        s = w * h
        r = w / float(h) if h != 0 else 1
        return np.array([cx, cy, s, r]).reshape((4, 1))

    def _from_x(self):
        cx, cy, s, r = self.kf.x[:4].flatten()
        w = np.sqrt(abs(s * r))
        h = s / w if w != 0 else 0
        return np.array([cx - w/2, cy - h/2, cx + w/2, cy + h/2])

    def predict(self):
        if self.kf.x[6] + self.kf.x[2] <= 0:
            self.kf.x[6] = 0
        self.kf.predict()
        self.age += 1
        self.time_since_update += 1
        return self._from_x()

    def update(self, bbox):
        self.kf.update(self._to_z(bbox))
        self.hits += 1
        self.hit_streak += 1
        self.time_since_update = 0

    def get_state(self):
        return self._from_x()


def iou_batch(bb_test, bb_gt):
    bb_gt = np.expand_dims(bb_gt, 0)
    bb_test = np.expand_dims(bb_test, 1)
    xx1 = np.maximum(bb_test[..., 0], bb_gt[..., 0])
    yy1 = np.maximum(bb_test[..., 1], bb_gt[..., 1])
    xx2 = np.minimum(bb_test[..., 2], bb_gt[..., 2])
    yy2 = np.minimum(bb_test[..., 3], bb_gt[..., 3])
    w = np.maximum(0, xx2 - xx1)
    h = np.maximum(0, yy2 - yy1)
    inter = w * h
    area_test = (bb_test[..., 2] - bb_test[..., 0]) * (bb_test[..., 3] - bb_test[..., 1])
    area_gt = (bb_gt[..., 2] - bb_gt[..., 0]) * (bb_gt[..., 3] - bb_gt[..., 1])
    union = area_test + area_gt - inter
    return inter / np.where(union == 0, 1e-9, union)


class SORTTracker:
    def __init__(self, max_age=30, min_hits=2, iou_threshold=0.25):
        self.max_age = max_age
        self.min_hits = min_hits
        self.iou_threshold = iou_threshold
        self.trackers = []
        self.frame_count = 0

    def update(self, detections):
        self.frame_count += 1

        predicted = []
        to_del = []
        for t, trk in enumerate(self.trackers):
            p = trk.predict()
            if np.any(np.isnan(p)):
                to_del.append(t)
            else:
                predicted.append(p)
        for t in reversed(to_del):
            self.trackers.pop(t)

        matched, unmatched_dets, unmatched_trks = self._associate(
            detections, predicted
        )

        for m in matched:
            self.trackers[m[1]].update(detections[m[0]])

        for i in unmatched_dets:
            self.trackers.append(KalmanBoxTracker(detections[i]))

        results = []
        to_del = []
        for t, trk in enumerate(self.trackers):
            state = trk.get_state()
            if (
                trk.time_since_update <= self.max_age and
                (trk.hit_streak >= self.min_hits or self.frame_count <= self.min_hits)
            ):
                results.append(np.concatenate([state, [trk.id]]))
            if trk.time_since_update > self.max_age:
                to_del.append(t)
        for t in reversed(to_del):
            self.trackers.pop(t)

        return np.array(results) if results else np.empty((0, 5))

    def _associate(self, detections, predictions):
        if len(predictions) == 0:
            return [], list(range(len(detections))), []
        if len(detections) == 0:
            return [], [], list(range(len(predictions)))

        iou_mat = iou_batch(np.array(detections), np.array(predictions))
        row_ind, col_ind = linear_sum_assignment(-iou_mat)

        matched = []
        unmatched_dets = []
        unmatched_trks = list(range(len(predictions)))

        matched_cols = set()
        for r, c in zip(row_ind, col_ind):
            if iou_mat[r, c] >= self.iou_threshold:
                matched.append([r, c])
                matched_cols.add(c)
            else:
                unmatched_dets.append(r)

        unmatched_trks = [c for c in range(len(predictions)) if c not in matched_cols]

        matched_det_ids = {m[0] for m in matched}
        for i in range(len(detections)):
            if i not in matched_det_ids:
                unmatched_dets.append(i)

        return matched, list(set(unmatched_dets)), unmatched_trks

## 5. Traffic Light Detector (HSV)

In [49]:
class TrafficLightDetector:

    def __init__(self):
        self.rois = [
            {
                "name": "Center Pole",
                "rect": (1800, 330, 30, 70)
            }
        ]

        self.hsv_thresholds = {
            "RED": {
                "lower1": np.array([0, 120, 80]),
                "upper1": np.array([10, 255, 255]),
                "lower2": np.array([165, 120, 80]),
                "upper2": np.array([179, 255, 255])
            },
            "YELLOW": {
                "lower": np.array([10, 100, 80]),
                "upper": np.array([35, 255, 255])
            },
            "GREEN": {
                "lower": np.array([36, 60, 60]),
                "upper": np.array([95, 255, 255])
            }
        }

        self.pixel_threshold = 30
        self.history_size = 9
        self.min_confirm_frames = 6
        self.kernel = np.ones((3, 3), np.uint8)

        self.state_histories = {
            roi["name"]: deque(maxlen=self.history_size)
            for roi in self.rois
        }
        self.stable_states = {
            roi["name"]: "OFF"
            for roi in self.rois
        }

    def detect(self, frame):
        detections = []
        for roi in self.rois:
            roi_name = roi["name"]
            x, y, w, h = roi["rect"]
            roi_frame = frame[y:y+h, x:x+w]
            if roi_frame.size == 0:
                continue
            state = self._detect_color(roi_frame, roi_name)
            detections.append({"name": roi_name, "bbox": (x, y, w, h), "state": state})
        return detections

    def get_state(self):
        return self.stable_states.get("Center Pole", "OFF")

    def _detect_color(self, roi_frame, roi_name):
        blurred = cv2.GaussianBlur(roi_frame, (5, 5), 0)
        hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)

        mask1 = cv2.inRange(hsv, self.hsv_thresholds["RED"]["lower1"], self.hsv_thresholds["RED"]["upper1"])
        mask2 = cv2.inRange(hsv, self.hsv_thresholds["RED"]["lower2"], self.hsv_thresholds["RED"]["upper2"])
        red_mask = cv2.bitwise_or(mask1, mask2)

        yellow_mask = cv2.inRange(hsv, self.hsv_thresholds["YELLOW"]["lower"], self.hsv_thresholds["YELLOW"]["upper"])
        green_mask = cv2.inRange(hsv, self.hsv_thresholds["GREEN"]["lower"], self.hsv_thresholds["GREEN"]["upper"])

        red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, self.kernel)
        yellow_mask = cv2.morphologyEx(yellow_mask, cv2.MORPH_OPEN, self.kernel)
        green_mask = cv2.morphologyEx(green_mask, cv2.MORPH_OPEN, self.kernel)

        red_pixels = cv2.countNonZero(red_mask)
        yellow_pixels = cv2.countNonZero(yellow_mask)
        green_pixels = cv2.countNonZero(green_mask)

        state = "OFF"
        if red_pixels > self.pixel_threshold and red_pixels > yellow_pixels and red_pixels > green_pixels:
            state = "RED"
        elif yellow_pixels > self.pixel_threshold and yellow_pixels > red_pixels and yellow_pixels > green_pixels:
            state = "YELLOW"
        elif green_pixels > self.pixel_threshold and green_pixels > red_pixels and green_pixels > yellow_pixels:
            state = "GREEN"

        return self._temporal_smoothing(roi_name, state)

    def _temporal_smoothing(self, roi_name, state):
        history = self.state_histories[roi_name]
        history.append(state)
        if history.count("RED") >= self.min_confirm_frames:
            self.stable_states[roi_name] = "RED"
        elif history.count("YELLOW") >= self.min_confirm_frames:
            self.stable_states[roi_name] = "YELLOW"
        elif history.count("GREEN") >= self.min_confirm_frames:
            self.stable_states[roi_name] = "GREEN"
        elif history.count("OFF") >= self.min_confirm_frames:
            self.stable_states[roi_name] = "OFF"
        return self.stable_states[roi_name]

## 6. ROI Config & Violation Logic

In [79]:
ROI_CONFIG = {
    "image_width": 1920,
    "image_height": 1200,
    "polygons": [
        {
            "id": 0,
            "points": [
                [338, 239],
                [916, 762],
                [1260, 586],
                [460, 171],
                [338, 233]
            ]
        }
    ]
}

DETECTION_POLYGON = np.array(ROI_CONFIG["polygons"][0]["points"], dtype=np.int32)

STOP_LINE = [
    DETECTION_POLYGON[1],
    DETECTION_POLYGON[2]
]

print("Detection polygon points:", DETECTION_POLYGON)
print("Stop line from:", STOP_LINE[0], "to:", STOP_LINE[1])

Detection polygon points: [[ 338  239]
 [ 916  762]
 [1260  586]
 [ 460  171]
 [ 338  233]]
Stop line from: [916 762] to: [1260  586]


In [80]:
class ViolationLogic:

    def __init__(self, polygon, stop_line):
        self.polygon = polygon
        self.stop_line_p1 = np.array(stop_line[0], dtype=float)
        self.stop_line_p2 = np.array(stop_line[1], dtype=float)

        self.track_history = {}
        self.prev_side = {}
        self.crossed_ids = set()
        self.violation_ids = set()
        self.violation_count = 0

        self.id_remap = {}
        self.remap_threshold = 60
        self.ghost_trackers = {}
        self.ghost_ttl = 60

    def _point_in_polygon(self, point):
        result = cv2.pointPolygonTest(
            self.polygon.reshape((-1, 1, 2)).astype(np.float32),
            (float(point[0]), float(point[1])),
            False
        )
        return result >= 0

    def _side_of_line(self, point):
        p = np.array(point, dtype=float)
        d = (
            (self.stop_line_p2[0] - self.stop_line_p1[0]) * (p[1] - self.stop_line_p1[1]) -
            (self.stop_line_p2[1] - self.stop_line_p1[1]) * (p[0] - self.stop_line_p1[0])
        )
        return np.sign(d)

    def _segment_intersects_line(self, p1, p2):
        s1 = self._side_of_line(p1)
        s2 = self._side_of_line(p2)
        if s1 == 0 or s2 == 0:
            return True
        return s1 != s2

    def resolve_id(self, raw_id, centroid):
        for ghost_id, ghost_data in list(self.ghost_trackers.items()):
            dist = np.linalg.norm(np.array(centroid) - np.array(ghost_data['last_pos']))
            if dist < self.remap_threshold:
                canonical_id = ghost_data['canonical_id']
                self.id_remap[raw_id] = canonical_id
                del self.ghost_trackers[ghost_id]
                return canonical_id
        if raw_id not in self.id_remap:
            self.id_remap[raw_id] = raw_id
        return self.id_remap[raw_id]

    def update_ghost(self, active_ids_with_pos, all_raw_ids):
        for raw_id, canonical_id in list(self.id_remap.items()):
            if raw_id not in all_raw_ids:
                if raw_id not in self.ghost_trackers:
                    last_pos = self.track_history.get(canonical_id, [None])[-1]
                    if last_pos is not None:
                        self.ghost_trackers[raw_id] = {
                            'canonical_id': canonical_id,
                            'last_pos': last_pos,
                            'ttl': self.ghost_ttl
                        }
        for gid in list(self.ghost_trackers.keys()):
            self.ghost_trackers[gid]['ttl'] -= 1
            if self.ghost_trackers[gid]['ttl'] <= 0:
                del self.ghost_trackers[gid]

    def _dist_to_stop_line(self, point):
            p = np.array(point, dtype=float)
            p1 = self.stop_line_p1
            p2 = self.stop_line_p2
            line_vec = p2 - p1
            line_len = np.linalg.norm(line_vec)
            if line_len == 0:
                return np.linalg.norm(p - p1)
            t = np.clip(np.dot(p - p1, line_vec) / (line_len ** 2), 0, 1)
            proj = p1 + t * line_vec
            return np.linalg.norm(p - proj)

    def update(self, tracks, light_state):
        current_raw_ids = set()
        current_active_ids = {}

        for track in tracks:
            x1, y1, x2, y2, raw_id = track
            raw_id = int(raw_id)
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            centroid = (cx, cy)
            current_raw_ids.add(raw_id)
            current_active_ids[raw_id] = centroid

            track_id = self.resolve_id(raw_id, centroid)

            if track_id not in self.track_history:
                self.track_history[track_id] = []
            self.track_history[track_id].append(centroid)

            curr_side = self._side_of_line(centroid)

            if track_id in self.prev_side and track_id not in self.crossed_ids:
                prev_s = self.prev_side[track_id]

                crossed = (
                    prev_s != 0 and
                    curr_side != 0 and
                    prev_s != curr_side
                )

                if crossed:
                    prev_pos = self.track_history[track_id][-2]
                    mid = (
                        (prev_pos[0] + centroid[0]) / 2,
                        (prev_pos[1] + centroid[1]) / 2
                    )
                    dist = self._dist_to_stop_line(mid)

                    prev_in_poly = self._point_in_polygon(prev_pos)
                    curr_in_poly = self._point_in_polygon(centroid)

                    if dist <= 150 and (prev_in_poly or curr_in_poly):
                        self.crossed_ids.add(track_id)
                        if light_state == "RED" and track_id not in self.violation_ids:
                            self.violation_ids.add(track_id)
                            self.violation_count += 1

            self.prev_side[track_id] = curr_side

        self.update_ghost(current_active_ids, current_raw_ids)
        return self.violation_count

## 7. Image Enhancement

In [83]:
def apply_clahe(frame, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_enhanced = clahe.apply(l)
    lab_enhanced = cv2.merge([l_enhanced, a, b])
    return cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)

def apply_unsharp(frame, sigma=1.0, strength=1.5):
    blurred = cv2.GaussianBlur(frame, (0, 0), sigma)
    return cv2.addWeighted(frame, 1 + strength, blurred, -strength, 0)

def apply_gamma(frame, gamma=1.5):
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype(np.uint8)
    return cv2.LUT(frame, table)

## 8. Main Pipeline

In [85]:
INPUT_VIDEO = '/content/drive/MyDrive/finpro-pkac/input/night_crowded.mp4'
OUTPUT_VIDEO = '/content/drive/MyDrive/finpro-pkac/output/night_crowded_violation.mp4'

KalmanBoxTracker.count = 0

detector = VehicleDetector(model_path='yolo11m.pt', conf=0.20, iou=0.4, device='cuda')
tracker = SORTTracker(max_age=10, min_hits=2, iou_threshold=0.25)
light_detector = TrafficLightDetector()
violation_logic = ViolationLogic(DETECTION_POLYGON, STOP_LINE)

cap = cv2.VideoCapture(INPUT_VIDEO)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (frame_w, frame_h))

COLOR_TRACK = (0, 255, 0)
COLOR_VIOLATION = (0, 0, 255)
COLOR_POLY = (255, 165, 0)
COLOR_STOPLINE = (0, 0, 255)

pbar = tqdm(total=total_frames, desc='Processing', unit='frame')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    light_detector.detect(frame)
    light_state = light_detector.get_state()

    enhanced_frame = apply_gamma(frame)
    enhanced_frame = apply_clahe(enhanced_frame)
    enhanced_frame = apply_unsharp(enhanced_frame)
    detections = detector.detect(enhanced_frame, polygon=DETECTION_POLYGON)

    if len(detections) > 0:
        tracks = tracker.update(detections[:, :4])
    else:
        tracks = tracker.update(np.empty((0, 4)))

    violation_count = violation_logic.update(tracks, light_state)

    cv2.polylines(frame, [DETECTION_POLYGON], isClosed=True, color=COLOR_POLY, thickness=2)
    cv2.line(frame,
             tuple(STOP_LINE[0].tolist()),
             tuple(STOP_LINE[1].tolist()),
             COLOR_STOPLINE, 2)

    for track in tracks:
        x1, y1, x2, y2, raw_id = track
        raw_id = int(raw_id)
        canonical_id = violation_logic.id_remap.get(raw_id, raw_id)
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        is_violation = canonical_id in violation_logic.violation_ids
        color = COLOR_VIOLATION if is_violation else COLOR_TRACK

        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(frame, f'ID:{canonical_id}', (int(x1), int(y1) - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    light_color_map = {"RED": (0, 0, 255), "YELLOW": (0, 255, 255), "GREEN": (0, 255, 0), "OFF": (128, 128, 128)}
    lc = light_color_map.get(light_state, (128, 128, 128))
    cv2.putText(frame, f'Light: {light_state}', (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, lc, 2)
    cv2.putText(frame, f'Violation Count: {violation_count}', (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

    out.write(frame)
    pbar.update(1)

pbar.close()
cap.release()
out.release()

print(f'Done. Violation Count: {violation_count}')
print(f'Output saved to: {OUTPUT_VIDEO}')

Processing:  99%|█████████▉| 4089/4127 [10:25<00:05,  6.54frame/s]

Done. Violation Count: 1
Output saved to: /content/drive/MyDrive/finpro-pkac/output/night_crowded_violation.mp4
